# Chat Memory

### Import Library

In [3]:
from haystack_experimental.chat_message_stores.in_memory import InMemoryChatMessageStore
from haystack_experimental.components.retrievers import ChatMessageRetriever
from haystack_experimental.components.writers import ChatMessageWriter
from haystack.dataclasses import ChatMessage
from haystack.components.joiners import ListJoiner
from haystack import Pipeline
from typing import List
from haystack.components.builders import ChatPromptBuilder, PromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.converters import OutputAdapter
from haystack.utils import Secret
from getpass import getpass
import os

In [4]:
os.environ["GROQ_API_KEY"] = getpass("Masukkan Groq API Key Anda: ")

In [5]:
os.environ["MONGO_CONNECTION_STRING"] = getpass("Masukkan MongoDB Connection String Anda: ")

### Membuat Pipeline

membuat chat tanpa memory

In [6]:
system_message = ChatMessage.from_system("You are a helpful assistant that answers questions based on the provided context.")
user_message_template = """
Answer the question based on the user query:
query:{{query}}
answer:
"""
user_message = ChatMessage.from_user(user_message_template)

In [7]:
pipeline = Pipeline()
pipeline.add_component("prompt_builder", ChatPromptBuilder(variables=["query"], required_variables=["query"]))
pipeline.add_component("generator", OpenAIChatGenerator(model="llama-3.3-70b-versatile", api_key=Secret.from_token(os.environ["GROQ_API_KEY"]), api_base_url="https://api.groq.com/openai/v1"))


pipeline.connect("prompt_builder.prompt", "generator.messages")

🚅 Components
  - prompt_builder: ChatPromptBuilder
  - generator: OpenAIChatGenerator
🛤️ Connections
  - prompt_builder.prompt -> generator.messages (List[ChatMessage])

In [8]:
while True:
    messages = [system_message, user_message]
    query = input("Please input your question or type 'exit' to quit.\n")
    if query.lower() == "exit":
        break
    res = pipeline.run(
        {
            "prompt_builder": {
                "query": query,
                "template":messages
            }
        }
    )
    print("AI Response:", res["generator"]["replies"][0].text)

AI Response: Baju musim panas biasanya terbuat dari bahan ringan dan sejuk, seperti katun atau linen, untuk membantu menjaga tubuh tetap sejuk dan nyaman selama cuaca panas. Beberapa contoh baju musim panas yang populer termasuk:

* Kaus oblong atau t-shirt
* Gaun sundress atau gaun panjang yang ringan
* Celana pendek atau rok pendek
* Baju polo atau baju lengan pendek
* Jumpsuit atau celana panjang yang ringan

Baju musim panas juga seringkali didesain dengan fitur-fitur seperti:

* Warna-warna cerah dan terang untuk memantulkan sinar matahari
* Desain yang longgar dan nyaman untuk memungkinkan sirkulasi udara yang baik
* Bahan yang dapat menyerap keringat dan cepat kering
* Aksesori seperti topi, kacamata hitam, dan sandal untuk melindungi diri dari sinar matahari langsung.
AI Response: A jumpsuit is a one-piece garment that covers the torso and legs, typically made of a single piece of fabric. It often has sleeves and may be fastened with a zipper, buttons, or other closures. Jumpsu

Membuat Pipeline dengan memory chat

mendefinisikan InMemoryChatMessageStore terlebih dahulu

In [9]:
memory_store = InMemoryChatMessageStore()
memory_retriever = ChatMessageRetriever(memory_store)
memory_writer = ChatMessageWriter(memory_store)

In [10]:
system_message = ChatMessage.from_system("You are a helpful assistant that answers questions based on the provided context.")
user_message_template = """
Answer the question based on the user query, please pay attention to the chat history:
chat_history:
{% for memory in memories %}
    {{memory.text}}
{% endfor %}

query:{{query}}
answer:
"""
user_message = ChatMessage.from_user(user_message_template)

In [11]:
pipeline = Pipeline()
pipeline.add_component("prompt_builder", ChatPromptBuilder(variables=["query","memories"], required_variables=["query","memories"]))
pipeline.add_component("generator", OpenAIChatGenerator(model="llama-3.3-70b-versatile", api_key=Secret.from_token(os.environ["GROQ_API_KEY"]), api_base_url="https://api.groq.com/openai/v1"))
pipeline.add_component("joiner", ListJoiner(List[ChatMessage]))
pipeline.add_component("memory_retriever", memory_retriever)
pipeline.add_component("memory_writer", memory_writer)

pipeline.connect("prompt_builder.prompt", "generator.messages")
pipeline.connect("generator.replies", "joiner")
pipeline.connect("joiner", "memory_writer")
pipeline.connect("memory_retriever", "prompt_builder.memories")

🚅 Components
  - prompt_builder: ChatPromptBuilder
  - generator: OpenAIChatGenerator
  - joiner: ListJoiner
  - memory_retriever: ChatMessageRetriever
  - memory_writer: ChatMessageWriter
🛤️ Connections
  - prompt_builder.prompt -> generator.messages (List[ChatMessage])
  - generator.replies -> joiner.values (List[ChatMessage])
  - joiner.values -> memory_writer.messages (List[ChatMessage])
  - memory_retriever.messages -> prompt_builder.memories (list[ChatMessage])

In [12]:
while True:
    messages = [system_message, user_message]
    query = input("Please input your question or type 'exit' to quit.\n")
    if query.lower() == "exit":
        break
    res = pipeline.run(
        data={
            "prompt_builder": {
                "query": query,
                "template":messages
            },
            "joiner":{
                "values": [ChatMessage.from_user(query)]
            },
            "memory_retriever": {
                "chat_history_id": "default"
            },
            "memory_writer": {
                "chat_history_id": "default"
            }
        },
        include_outputs_from=["generator"]
    )
    # print(res)
    print("AI Response:", res["generator"]["replies"][0].text)

AI Response: Baju musim panas biasanya terbuat dari bahan yang ringan dan sejuk, seperti katun, linen, atau rayon. Beberapa tips untuk memilih baju musim panas yang nyaman adalah:

* Pilih baju dengan warna cerah untuk memantulkan sinar matahari
* Pilih baju dengan potongan yang longgar untuk memungkinkan sirkulasi udara
* Pilih baju dengan bahan yang dapat menyerap keringat dengan baik
* Pilih baju dengan model yang sederhana dan tidak terlalu banyak detail untuk menghindari panas

Beberapa contoh baju musim panas yang populer adalah:
* Kemeja katun
* Gaun sundress
* Celana pendek
* Baju kaos

Apakah Anda memiliki preferensi tertentu untuk baju musim panas?
AI Response: Beberapa pilihan baju musim panas yang bagus adalah kemeja katun, gaun sundress, celana pendek, dan baju kaos. Baju-baju tersebut terbuat dari bahan ringan dan sejuk, sehingga nyaman dikenakan saat musim panas. Anda juga bisa mempertimbangkan pilihan baju dengan warna cerah, potongan longgar, dan bahan yang dapat menye